# XAS scan aggregates

Inspect aggregates produced by `compute_xas_aggregates.py`. The file holds per-(nominal photon energy, GMD bin) means of the VLS spectrum and the GMD; this notebook reads them back and plots:

1. shot counts per bin
2. mean GMD per bin
3. mean VLS spectrum per energy (pixels x GMD bin, one subplot per energy)
4. XAS(E) for each GMD bin on one axis

XAS per bin is `G[E,g] / sum_pixels(A[E,g])`, i.e. sum(GMD)/sum(VLS) over the shots in that bin.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "analysis" / "scripts").exists():
        repo_root = p
        break
if repo_root is None:
    raise RuntimeError("Could not locate repository root containing analysis/scripts")
sys.path.insert(0, str(repo_root / "analysis" / "scripts"))

import config as path_config
from compute_aggregates import load_aggregates

## Load aggregates

Point `AGG_PATH` at the H5 file written by `compute_xas_aggregates.py`. The default below assumes the convention `<COMBINED_DIR>/run<RUN_NO>_xas_aggregates.h5`.

In [ ]:
RUN_NO = 58780
AGG_PATH = Path(path_config.COMBINED_DIR) / f"run{RUN_NO}_xas_aggregates.h5"

agg = load_aggregates(AGG_PATH)
if agg.mode != "xas_scan":
    raise RuntimeError(f"expected mode='xas_scan', got {agg.mode!r}")

energies   = agg.nominal_energies        # (N_E,)
gmd_edges  = agg.gmd_edges               # (N_GMD+1,)
gmd_cents  = agg.gmd_centres             # (N_GMD,)
vls_pixels = agg.vls_pixels              # (n_pixels,)
n_per_bin  = agg.n_per_bin               # (N_E, N_GMD)
G_mean     = agg.G                       # (N_E, N_GMD)
A_mean     = agg.A                       # (N_E, N_GMD, n_pixels)

N_E, N_GMD = n_per_bin.shape
n_pixels = vls_pixels.size
print(f"file:    {AGG_PATH}")
print(f"run:     {agg.metadata.get('run_no')}")
print(f"shape:   N_E={N_E}, N_GMD={N_GMD}, n_pixels={n_pixels}")
print(f"E range: {energies[0]:.2f} .. {energies[-1]:.2f} eV")
print(f"GMD edges: {gmd_edges}")
print(f"total shots: {int(n_per_bin.sum())}")
print(f"detected sections: {agg.metadata.get('n_sections_detected')} "
      f"(used {agg.metadata.get('n_sections_used')})")

## Shot counts per bin

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.5))
im = ax.imshow(
    n_per_bin.T,
    aspect="auto",
    origin="lower",
    extent=[energies[0], energies[-1], 0, N_GMD],
    cmap="viridis",
)
fig.colorbar(im, ax=ax, label="shots / bin")
ax.set_xlabel("nominal photon energy (eV)")
ax.set_ylabel("GMD bin index")
ax.set_yticks(np.arange(N_GMD) + 0.5)
ax.set_yticklabels([f"[{gmd_edges[i]:.2f}, {gmd_edges[i+1]:.2f})" for i in range(N_GMD)])
ax.set_title(f"Shot count per (energy, GMD) bin - run {agg.metadata.get('run_no')}")
fig.tight_layout()
plt.show()

# Marginal: shots per energy.
fig, ax = plt.subplots(figsize=(7.0, 3.0))
ax.plot(energies, n_per_bin.sum(axis=1), "o-", color="tab:blue")
ax.set_xlabel("nominal photon energy (eV)")
ax.set_ylabel("shots per energy")
ax.set_title("Shot count vs nominal photon energy (summed over GMD bins)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Mean GMD per bin

Per-bin mean of the GMD intensity (`G`). Empty bins are NaN and show as blank cells.

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.5))
im = ax.imshow(
    G_mean.T,
    aspect="auto",
    origin="lower",
    extent=[energies[0], energies[-1], 0, N_GMD],
    cmap="magma",
)
fig.colorbar(im, ax=ax, label="mean GMD (uJ)")
ax.set_xlabel("nominal photon energy (eV)")
ax.set_ylabel("GMD bin")
ax.set_yticks(np.arange(N_GMD) + 0.5)
ax.set_yticklabels([f"[{gmd_edges[i]:.2f}, {gmd_edges[i+1]:.2f})" for i in range(N_GMD)])
ax.set_title("Mean GMD per (energy, GMD) bin")
fig.tight_layout()
plt.show()

# Mean GMD vs energy, one line per GMD bin.
fig, ax = plt.subplots(figsize=(7.0, 3.4))
cmap = plt.get_cmap("viridis")
for g in range(N_GMD):
    ax.plot(energies, G_mean[:, g], "o-", color=cmap(g / max(N_GMD - 1, 1)),
            label=f"[{gmd_edges[g]:.2f}, {gmd_edges[g+1]:.2f}) uJ")
ax.set_xlabel("nominal photon energy (eV)")
ax.set_ylabel("mean GMD (uJ)")
ax.set_title("Mean GMD vs energy")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

## VLS aggregate per energy

One subplot per nominal photon energy. Each subplot shows the mean VLS spectrum laid out as `(GMD bin, pixel)` so the GMD axis (intensity dependence) runs vertically and the spectral axis (pixel) horizontally. Empty bins are blank.

In [ ]:
# Grid layout: roughly square.
ncols = int(np.ceil(np.sqrt(N_E)))
nrows = int(np.ceil(N_E / ncols))

# Use a shared vmin/vmax across panels for visual comparability.
finite_mask = np.isfinite(A_mean)
if finite_mask.any():
    vmin, vmax = np.nanpercentile(A_mean[finite_mask], [2, 98])
else:
    vmin, vmax = 0.0, 1.0

fig, axes = plt.subplots(nrows, ncols, figsize=(2.0 * ncols, 1.6 * nrows),
                         sharex=True, sharey=True, squeeze=False)
for k in range(nrows * ncols):
    r, c = divmod(k, ncols)
    ax = axes[r][c]
    if k >= N_E:
        ax.axis("off")
        continue
    img = A_mean[k]  # (N_GMD, n_pixels)
    im = ax.imshow(
        img,
        aspect="auto",
        origin="lower",
        extent=[vls_pixels[0], vls_pixels[-1], 0, N_GMD],
        cmap="viridis",
        vmin=vmin, vmax=vmax,
    )
    ax.set_title(f"{energies[k]:.1f} eV", fontsize=8)
    if r == nrows - 1:
        ax.set_xlabel("VLS pixel", fontsize=8)
    if c == 0:
        ax.set_ylabel("GMD bin", fontsize=8)
fig.suptitle("Mean VLS spectrum per (energy, GMD bin)", fontsize=11)
fig.colorbar(im, ax=axes, shrink=0.7, label="mean intensity (arb.)")
plt.show()

## XAS(E) per GMD bin

Per-bin XAS = `<GMD> / sum_pixels(<VLS>)`. This is equivalent to `sum_shots(GMD) / sum_shots(sum_pixels(VLS))` because shot counts cancel. One line per GMD bin.

In [ ]:
vls_sum = np.nansum(A_mean, axis=2)        # (N_E, N_GMD)
with np.errstate(divide="ignore", invalid="ignore"):
    xas = G_mean / vls_sum                  # (N_E, N_GMD)

fig, ax = plt.subplots(figsize=(8.0, 4.5))
cmap = plt.get_cmap("viridis")
for g in range(N_GMD):
    label = f"[{gmd_edges[g]:.2f}, {gmd_edges[g+1]:.2f}) uJ"
    ax.plot(energies, xas[:, g], "o-",
            color=cmap(g / max(N_GMD - 1, 1)),
            lw=1.4, label=label)
ax.set_xlabel("nominal photon energy (eV)")
ax.set_ylabel("XAS = <GMD> / sum_pix(<VLS>)")
ax.set_title(f"Run {agg.metadata.get('run_no')}: XAS(E) per GMD bin")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, title="GMD bin", ncol=2)
fig.tight_layout()
plt.show()

# Table for the record.
print(f"{'E (eV)':>9s}" + "".join(f"  bin{g} XAS  " for g in range(N_GMD)))
print("-" * (9 + 11 * N_GMD))
for k in range(N_E):
    row = f"{energies[k]:9.2f}"
    for g in range(N_GMD):
        row += f"  {xas[k, g]:9.4g}"
    print(row)